In [5]:
import fitz
import pytesseract
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [ ]:
# Load model
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

In [ ]:

# Convert page of PDF to image
doc = fitz.open('../data/CO715-1.pdf')
page = doc[6]
pix = page.get_pixmap(dpi=300)
pix.save('page.png')

In [ ]:

# Run OCR
image = Image.open('page.png').convert('RGB')
pixel_values = processor(image, return_tensors='pt').pixel_values
generated_ids = model.generate(pixel_values)
text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(text)

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [ ]:
# Process in batches to avoid OOM
BATCH_SIZE = 16
results = []

for i in range(0, len(crops), BATCH_SIZE):
    batch = crops[i : i + BATCH_SIZE]
    
    pixel_values = processor(
        batch, 
        return_tensors='pt', 
        padding=True
    ).pixel_values
    
    # Move to GPU if available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    pixel_values = pixel_values.to(device)
    model.to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    
    batch_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
    results.extend(batch_text)
    print(f"Processed batch {i//BATCH_SIZE + 1}/{(len(crops)-1)//BATCH_SIZE + 1}")

full_text = ' '.join(results)
print(full_text)